# Final Project: Modern Statistical Data Analysis 52311
Omer Sutovsky and Ido Ravid

In [1]:
# Imports
import os, glob
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.stats import spearmanr
from scipy import stats
import contextily as ctx
from pyproj import Transformer
from bidi.algorithm import get_display
from PIL import Image

_CODE_DIR       = os.getcwd()
DATA_DIR        = os.path.normpath(os.path.join(_CODE_DIR, '..', 'data'))
PLOTS_DIR_UNSUP = os.path.normpath(os.path.join(_CODE_DIR, '..', 'plots', 'unsupervised'))
PLOTS_DIR_HYP   = os.path.normpath(os.path.join(_CODE_DIR, '..', 'plots', 'hypothesis'))
os.makedirs(PLOTS_DIR_UNSUP, exist_ok=True)
os.makedirs(PLOTS_DIR_HYP, exist_ok=True)

---
## Section 1 - Supervised Learning

---
## Section 2 - Unsupervised Learning

### Q2 - Merging the crimes and cities tables

In [2]:
########## Auxiliary - assisted by Claude ##########

def load_and_merge():
    CRIMES_FILE = os.path.join(DATA_DIR, '2012026154636_2025_hangashat_meyda_plili.xlsx')
    CITIES_FILE = os.path.join(DATA_DIR, 'cities_israel.xls')

    columns_to_read = ['YeshuvKod', 'StatisticGroupKod', 'StatisticTypeKod']
    df = pd.read_csv(CRIMES_FILE, usecols=columns_to_read, encoding='utf-8-sig')

    df_cities = pd.read_excel(CITIES_FILE, usecols=['symbol', 'coordinates', 'population'])
    df_cities = df_cities.rename(columns={'symbol': 'YeshuvKod'})

    df_counts = df.groupby(['YeshuvKod', 'StatisticTypeKod']).size().unstack(fill_value=0)
    df_merged = pd.merge(df_counts.reset_index(), df_cities, on='YeshuvKod', how='left')
    return df_merged

df_merged_raw = load_and_merge()
df_merged_raw.head(3)

,YeshuvKod,106,203,204,206,207,215,216,220,401,...,727,804,10027,726,218,10004,10015,306,population,coordinates
0,26.0,4,1,5,28,9,16,6,4,2,...,0,0,0,0,0,0,0,0,2906.0,2.510576e+09
1,28.0,2,0,3,50,19,14,4,8,1,...,0,0,0,0,0,0,0,0,11857.0,1.852464e+09
2,29.0,2,0,1,3,1,3,0,2,0,...,0,0,0,0,0,0,0,0,1654.0,2.566377e+09


### Q3 - Graph Laplacian

In [3]:
########## Auxiliary - assisted by Claude ##########

def save_matrix(matrix, filename):
    with open(filename, 'wb') as f: pickle.dump(matrix, f)

def load_matrix(filename):
    with open(filename, 'rb') as f: return pickle.load(f)

We elaborated more on the preprocessing part in the writeup:

In [4]:
BAD_GROUPS = {-1, 1200, 1300}

def preprocess(df_merged):
    # drop unknown settlement
    df_merged = df_merged.dropna(subset=['YeshuvKod'])

    # drop prblematic data categories as mentioned in the writeup: -1, 1200, 1300
    crime_cols_all = [c for c in df_merged.columns if isinstance(c, int) and c not in BAD_GROUPS]
    df_merged = df_merged[['YeshuvKod'] + crime_cols_all + ['coordinates', 'population']]

    # keep cities with known population and coords
    df_merged['population'] = pd.to_numeric(df_merged['population'], errors='coerce')
    df_merged['coordinates'] = pd.to_numeric(df_merged['coordinates'], errors='coerce')
    df_merged = df_merged.dropna(subset=['population', 'coordinates'])
    df_merged = df_merged[df_merged['population'] >= 2000]
    df_merged = df_merged.set_index('YeshuvKod')

    # Parse ITM coordinate integer into x / y
    coord_str = df_merged['coordinates'].astype(int).astype(str).str.zfill(10)
    df_merged['x'] = coord_str.str[:5].astype(float)
    df_merged['y'] = coord_str.str[5:10].astype(float)
    df_merged = df_merged.drop(columns=['coordinates'])

    crime_cols = [c for c in df_merged.columns if isinstance(c, int)]
    print(f'Cities after preprocessing: {len(df_merged)}')
    print(f'Crime subtypes: {len(crime_cols)}')
    return df_merged, crime_cols



Here we calculate the embedding matrix V, the weight matrix W and the diagonal matrix D, and finally the Laplacian L. The similarity and normalization logic we selected is in the writeup as well.

In [5]:
## Step 3 - compute Laplacian

# Create embedding matrix:
def compute_embedding_matrix(C, pop):
    # noarmlize by population, resphape to (|crime_cols|, 1)
    V = C / pop[:, np.newaxis]
    # L2 normalization per row, so we compare crime mix profile
    row_norms = np.linalg.norm(V, axis=1, keepdims=True)
    row_norms[row_norms == 0] = 1
    V = V / row_norms
    return V

# calculating sigma based on knn like recommended in "A Tutorial on Spectral Clustering" paper we learned
def compute_sigmas(V, k=7):
    n = V.shape[0]
    sigmas = np.zeros(n)
    for i in range(n):
        dists = np.array([np.sum((V[i] - V[j]) ** 2) ** 0.5 for j in range(n) if j != i])
        dists.sort()
        sigmas[i] = dists[k - 1]
    return sigmas

# Compute weight matrix W:
def compute_LW_matrices(V, sigmas, n):
    W = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            diff = V[i]-V[j]
            # Kernel
            dist_squared =np.sum(diff**2)
            W[i,j] = np.exp(-dist_squared / (sigmas[i] * sigmas[j]))
            W[j,i] = W[i,j]
    np.fill_diagonal(W, 0)
    # Copmute L and D
    D= np.zeros((n,n))
    for i in range(n):
        cur_sum =0
        for j in range(n):
            cur_sum+= W[i,j]
        D[i,i] =  cur_sum
    L = D-W
    return L,W

df_proc, crime_cols = preprocess(df_merged_raw)
n   = len(df_proc)
C   = df_proc[crime_cols].values.astype(float)
pop = df_proc['population'].values
V   = compute_embedding_matrix(C, pop)
sigmas = compute_sigmas(V)

W_PATH = os.path.join(_CODE_DIR, 'W.pkl')
L_PATH = os.path.join(_CODE_DIR, 'L.pkl')
if os.path.exists(W_PATH) and os.path.exists(L_PATH):
    W = load_matrix(W_PATH)
    L = load_matrix(L_PATH)
else:
    L, W = compute_LW_matrices(V, sigmas, n)
    save_matrix(W, W_PATH)
    save_matrix(L, L_PATH)

Cities after preprocessing: 206
Crime subtypes: 152


### Q4 - Eigenvectors

In [6]:
########## Auxiliary - assisted by Claude ##########

_itm_to_webmercator = Transformer.from_crs("EPSG:2039", "EPSG:3857", always_xy=True)

_LABELED_CITIES = [
    (19423, 38515, 'אילת'),
    (17966, 57357, 'באר שבע'),
    (22086, 63221, 'ירושלים'),
    (16691, 63377, 'אשדוד'),
    (18027, 66485, 'תל אביב-יפו'),
    (18726, 69145, 'נתניה'),
    (20112, 74546, 'חיפה'),
    (24942, 74369, 'טבריה'),
    (25383, 79074, 'קריית שמונה'),
]

def plot_cities(x, y, values, title, filename):
    mx, my = _itm_to_webmercator.transform(x * 10, y * 10)
    abs_max = np.percentile(np.abs(values), 95)
    norm = plt.Normalize(vmin=-abs_max, vmax=abs_max)
    fig, ax = plt.subplots(figsize=(8, 20))
    ax.scatter(mx, my, c=np.clip(values, -abs_max, abs_max), cmap='coolwarm', norm=norm,
               s=50, edgecolors='black', linewidths=0.3, alpha=0.9, zorder=3)
    for cx, cy, name in _LABELED_CITIES:
        wmx, wmy = _itm_to_webmercator.transform(cx * 10, cy * 10)
        ax.annotate(get_display(name), xy=(wmx, wmy), fontsize=12, rotation=90,
                    ha='center', va='bottom', zorder=4, color='#111111',
                    xytext=(0, 5), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, zoom=8)
    ax.set_axis_off()
    plt.tight_layout(pad=0)
    tmp_map = os.path.join(PLOTS_DIR_UNSUP, '_tmp_' + filename)
    plt.savefig(tmp_map, dpi=150, bbox_inches='tight')
    plt.close()
    rotated = Image.open(tmp_map).rotate(270, expand=True)
    os.remove(tmp_map)
    fig, (ax_map, ax_cb) = plt.subplots(1, 2, figsize=(20, 8),
                                         gridspec_kw={'width_ratios': [20, 1]})
    ax_map.imshow(rotated)
    ax_map.set_axis_off()
    ax_map.set_title(title, fontsize=16, pad=10)
    abs_max = np.percentile(np.abs(values), 95)
    norm = plt.Normalize(vmin=-abs_max, vmax=abs_max)
    sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=norm)
    plt.colorbar(sm, cax=ax_cb)
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(os.path.join(PLOTS_DIR_UNSUP, filename), dpi=150, bbox_inches='tight')
    plt.close()

In [7]:
## Step 4 - EV calculation
def calc_print_EV(L, x, y, city_names):
    # get eigenvectors and eigenvalues
    eigenvals, eigenvecs = np.linalg.eigh(L)

    for t in range(1, 11): # skipping the trivial 0-eigenvalue vector:
        ev = eigenvecs[:, t]
        title = f'Eigenvector {t} (λ={eigenvals[t]:.4f})'
        plot_cities(x, y, ev, title=title, filename=f'eigenvector_{t}.png')
    return eigenvals, eigenvecs

x, y = df_proc['x'].values, df_proc['y'].values
df_cities_names = pd.read_excel(os.path.join(DATA_DIR, 'cities_israel.xls'),
                                 usecols=['symbol', 'שם יישוב'])
kod_to_name = dict(zip(df_cities_names['symbol'], df_cities_names['שם יישוב']))
city_names = [kod_to_name.get(k, str(int(k))) for k in df_proc.index]

eigenvals, eigenvecs = calc_print_EV(L, x, y, city_names)

Here we can see the eigenvectors of each eigenvalue, sorted from largest to smallest. our analysis about which cities correspond to which vectors is in the writeup.

### Q5 - Spectral Clustering

In [8]:
## Step 5 - spectral clustering
def spectral_clustering(eigenvecs, k):
    v = eigenvecs[:, 1:k + 1]
    kmeans = KMeans(n_clusters=k, random_state=23)
    labels = kmeans.fit_predict(v)
    return labels

clusters = spectral_clustering(eigenvecs, 5)

In [9]:
########## Auxiliary - assisted by Claude ##########

def plot_eigengap(eigenvalues, n=20):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, n + 1), eigenvalues[1:n + 1], 'o-')
    ax.set_xlabel('Index')
    ax.set_ylabel('Eigenvalue')
    ax.set_title('Eigenvalue spectrum (eigengap)')
    ax.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR_UNSUP, 'eigengap.png'), dpi=150)
    plt.close()

plot_eigengap(eigenvals)

def plot_clusters(x, y, labels, title, filename):
    mx, my = _itm_to_webmercator.transform(x * 10, y * 10)
    k = len(np.unique(labels))
    cmap = plt.cm.get_cmap('tab10', k)
    fig, ax = plt.subplots(figsize=(8, 20))
    ax.scatter(mx, my, c=labels, cmap=cmap, vmin=0, vmax=k - 1,
               s=50, edgecolors='black', linewidths=0.3, alpha=0.9, zorder=3)
    for cx, cy, name in _LABELED_CITIES:
        wmx, wmy = _itm_to_webmercator.transform(cx * 10, cy * 10)
        ax.annotate(get_display(name), xy=(wmx, wmy), fontsize=12, rotation=90,
                    ha='center', va='bottom', zorder=4, color='#111111',
                    xytext=(0, 5), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, zoom=8)
    ax.set_axis_off()
    plt.tight_layout(pad=0)
    tmp_map = os.path.join(PLOTS_DIR_UNSUP, '_tmp_' + filename)
    plt.savefig(tmp_map, dpi=150, bbox_inches='tight')
    plt.close()
    rotated = Image.open(tmp_map).rotate(270, expand=True)
    os.remove(tmp_map)
    fig, (ax_map, ax_cb) = plt.subplots(1, 2, figsize=(20, 8),
                                         gridspec_kw={'width_ratios': [20, 1]})
    ax_map.imshow(rotated)
    ax_map.set_axis_off()
    ax_map.set_title(title, fontsize=18)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=k - 1))
    cbar = plt.colorbar(sm, cax=ax_cb, ticks=range(k))
    cbar.set_ticklabels([f'Cluster {i+1}' for i in range(k)])
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR_UNSUP, filename), dpi=150)
    plt.close()

plot_clusters(x, y, clusters, title='Spectral Clustering (k=5)', filename='clusters.png')

/var/folders/l4/gpnymly96vgbjsb7l6tnl0y40000gq/T/ipykernel_46721/1133325625.py:19: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab10', k)


Here we plotted the eigengap graph and the spectral clusters for K=5. our selection of K=5 is justified in the writeup, and our analysis of the clusters as well.

### Q6 - Feature Correlations

In [10]:
########## Auxiliary - assisted by Claude ##########

_df_religion = pd.read_excel(os.path.join(DATA_DIR, 'cities_israel.xls'),
                              usecols=['symbol', 'דת יישוב']).rename(columns={'symbol': 'YeshuvKod'})
_RELIGION_MAP = {1: 'Jewish', 2: 'Arab', 3: 'Druze/Other', 4: 'Mixed'}

_CITY_FEATURES = ['population', 'מחוז', 'דת יישוב', 'מעמד מונציפאלי', 'גובה ']
_FEATURE_LABELS = ['Population', 'District', 'Religion', 'Municipal status', 'Elevation']
_df_features = pd.read_excel(os.path.join(DATA_DIR, 'cities_israel.xls'),
                              usecols=['symbol'] + _CITY_FEATURES).rename(columns={'symbol': 'YeshuvKod'})

def print_correlation_table(labels, rows):
    print(f"{'Feature':<20}", end='')
    for label in labels:
        print(f"  {label:>6}", end='')
    print()
    print('-' * (20 + 8 * len(labels)))
    for feat_label, values, stars in rows:
        print(f"{feat_label:<20}", end='')
        for r, s in zip(values, stars):
            print(f"  {r:+.2f}{s}", end='')
        print()
    print()

In [11]:
## Step 6 - analyzing ethnicity and correlate eigenvectors with city features
def analyze_ethnicity(clusters, yeshuv_kods):
    # pair city with its kod, join the religion df
    df = pd.DataFrame({'cluster': clusters, 'YeshuvKod': yeshuv_kods})
    df = df.merge(_df_religion, on='YeshuvKod', how='left')
    # convert codes to labels
    df['religion'] = df['דת יישוב'].map(_RELIGION_MAP).fillna('Unknown')
    # count cluster-religion pairs
    table = df.groupby(['cluster', 'religion']).size().unstack(fill_value=0)
    table.index = [f'Cluster {i+1}' for i in table.index]
    print(table.to_string())
    print()

def correlate_features(eigenvecs, yeshuv_kods):
    # load df
    df = pd.DataFrame({'YeshuvKod': yeshuv_kods}).merge(_df_features, on='YeshuvKod', how='left')
    ev_labels = [f'EV{t}' for t in range(1, 6)]
    rows = []
    for col, label in zip(_CITY_FEATURES, _FEATURE_LABELS):
        feat = df[col].values
        # mask missing values from correlation computation
        mask = ~np.isnan(feat)
        rs, stars = [], []
        # calculate spearman
        for t in range(1, 6):
            r, p = spearmanr(eigenvecs[mask, t], feat[mask])
            rs.append(r)
            # check for significance ( uncorrected for BH like in next part)
            stars.append('*' if p < 0.05 else ' ')
        rows.append((label, rs, stars))
    print_correlation_table(ev_labels, rows)

analyze_ethnicity(clusters, list(df_proc.index))
correlate_features(eigenvecs, list(df_proc.index))

religion   Arab  Jewish  Mixed
Cluster 1     2       3      0
Cluster 2    15      86      7
Cluster 3    64       1      0
Cluster 4     1       0      0
Cluster 5     0      26      1

Feature                  EV1     EV2     EV3     EV4     EV5
------------------------------------------------------------
Population            -0.42*  -0.15*  -0.00   -0.09   -0.30*
District              -0.39*  +0.04   +0.11   +0.03   +0.07 
Religion              +0.70*  +0.28*  -0.30*  +0.10   -0.11 
Municipal status      +0.48*  +0.28*  -0.23*  +0.12   +0.05 
Elevation             +0.38*  -0.03   +0.16*  -0.00   +0.44*



Results analysis is in the writeup

---
## Section 3 - Hypothesis Testing / Multiple Testing

Our Hypothesis testing was around differences in crime patterns between 2022 and 2025. Claude here assisted us with loading the tables and plotting the p-values. we used the crime rate per-year per-city as a statistic, normalized per-capita, and then performed standard two-sample t-testing. Afterwards we corrected for BH. Full explanation and analysis are in the writeup.

In [12]:
########## Auxiliary - assisted by Claude ##########

def _load_one_file(filepath):
    cols = ['Year', 'YeshuvKod', 'StatisticGroupKod', 'StatisticTypeKod', 'StatisticType']
    # try csv first; fall back to xls/xlsx
    try:
        df = pd.read_csv(filepath, usecols=cols, encoding='utf-8-sig')
    except Exception:
        df = pd.read_excel(filepath, usecols=cols)
    return df

def load_yearly_rates(filepaths):
    # load files, filter bad groups and unknown cities, join population >= 2000
    frames = [_load_one_file(p) for p in filepaths]
    df = pd.concat(frames, ignore_index=True)
    df = df[df['Year'].isin([2022, 2025])]
    df = df[~df['StatisticGroupKod'].isin(BAD_GROUPS)]
    df = df.dropna(subset=['YeshuvKod'])

    df_cities = pd.read_excel(os.path.join(DATA_DIR, 'cities_israel.xls'),
                               usecols=['symbol', 'population'])
    df_cities = df_cities.rename(columns={'symbol': 'YeshuvKod'})
    df_cities['population'] = pd.to_numeric(df_cities['population'], errors='coerce')
    df_cities = df_cities[df_cities['population'] >= 2000]

    df = df.merge(df_cities[['YeshuvKod', 'population']], on='YeshuvKod', how='inner')
    return df

def plot_pvalues(df, bonf_threshold):
    # df is sorted output of run_t_test_with_bh (ascending by p_value)
    # style follows Ex2/q2.py: rank on x, p-value on log-scale y, horizontal threshold lines
    m = len(df)
    sorted_p = df['p_value'].values  # already sorted ascending
    bh_thresh = df.loc[df['reject_bh'], 'p_value'].max() if df['reject_bh'].any() else 0
    bonf_thresh = bonf_threshold

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(range(1, m + 1), sorted_p, marker='.', markersize=4, linewidth=0.8,
            color='steelblue', label='p-values')
    ax.axhline(bonf_thresh, color='orange', linestyle='--',
               label=f'Bonferroni ({bonf_thresh:.5f})')
    ax.axhline(bh_thresh, color='black', linestyle='--',
               label=f'BH threshold ({bh_thresh:.5f})')
    # list significant crime types in bottom-right as a text block
    sig = df[df['reject_bh']].reset_index(drop=True)
    label_text = 'Significant crime types:\n' + '\n'.join(
        get_display(f"{i+1}. {row['StatisticType']}")
        for i, row in sig.iterrows()
    )
    ax.text(0.98, 0.02, label_text, transform=ax.transAxes,
            fontsize=7, va='bottom', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8, ec='lightgrey'))
    ax.set_yscale('log')
    ax.set_xlabel('Rank')
    ax.set_ylabel('p-value')
    ax.set_title('2022 vs 2025: crime type p-values')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR_HYP, 'pvalues.png'), dpi=150)
    plt.close()

In [13]:
########## Multiple Hypothesis Testing ##########

def compute_city_rates(df):
    # aggregate (type, city, year)
    counts = (
        df.groupby(['StatisticTypeKod', 'StatisticType', 'YeshuvKod', 'population', 'Year'])
        .size()
        .rename('crime_count')
        .reset_index()
    )
    # compute per-capita crime rate
    counts['crime_rate'] = counts['crime_count'] / counts['population']
    return counts

def run_t_test(counts):
    results = []
    for type_kod, group in counts.groupby('StatisticTypeKod'):
        # each entry is one city's per-capita rate for the specific crime type
        rates_2022 = group[group['Year'] == 2022]['crime_rate'].values
        rates_2025 = group[group['Year'] == 2025]['crime_rate'].values

        # skip rare types with too few cities to estimate variance
        if len(rates_2022) < 2 or len(rates_2025) < 2:
            continue

        #calculating welch t_test statistic and accordingly the pval, without assuming equal varaince
        T_i, p_i = stats.ttest_ind(rates_2022, rates_2025, equal_var=False)
        results.append({
            'StatisticTypeKod': type_kod, 'StatisticType': group['StatisticType'].iloc[0], 'mean_2022': rates_2022.mean(),
            'mean_2025': rates_2025.mean(), 'effect': rates_2025.mean() - rates_2022.mean(), 'T_i': T_i, 'p_value': p_i,
        })
    return results

In [14]:
def bh_correct(results, alpha=0.05):
    df = pd.DataFrame(results)
    # calc bonferroni
    m = len(df)
    bonf_threshold = alpha / m

    ##  calc BH:
    # sort pvals:
    df = df.sort_values('p_value').reset_index(drop=True)
    # calc bf threshold
    df['bh_threshold'] = (df.index + 1) * alpha / m
    # find k corresponding to threshold
    k = df[df['p_value'] <= df['bh_threshold']].index.max()
    # reject all hypotheses with rank <= k_max
    df['reject_bh'] = False
    if not pd.isna(k):
        df.loc[:k, 'reject_bh'] = True

    df['reject_bonferroni'] = df['p_value'] < bonf_threshold
    return df, bonf_threshold

In [15]:
files = sorted(glob.glob(os.path.join(DATA_DIR, '2012026154636_2022*.csv'))) +         sorted(glob.glob(os.path.join(DATA_DIR, '2012026154636_2025*.xlsx')))
counts = compute_city_rates(load_yearly_rates(files))
results = run_t_test(counts)
df_results, bonf_threshold = bh_correct(results)
plot_pvalues(df_results, bonf_threshold)

sig_bh   = df_results[df_results['reject_bh']]
sig_bonf = df_results[df_results['reject_bonferroni']]
print(f"Total crime types tested: {len(df_results)}")
print(f"Bonferroni significant: {len(sig_bonf)}")
print(f"BH significant: {len(sig_bh)}")
sig_bh[['StatisticType', 'mean_2022', 'mean_2025', 'effect', 'p_value']]

Total crime types tested: 136
Bonferroni significant: 3
BH significant: 9


,StatisticType,mean_2022,mean_2025,effect,p_value
0,שמוש בסמים מסוכנים,0.001379,0.000436,-0.000943,5.837166e-19
1,מרמה ועושק,0.001003,0.001441,0.000438,4.917264e-11
2,הפרעות במהלך בחירות תקין,0.000040,0.000004,-0.000036,5.278859e-06
3,סחיטה,0.000283,0.000426,0.000143,5.933150e-04
4,גידולייצור והפקת-סמים,0.000129,0.000074,-0.000055,1.000499e-03
5,מעשה מגונה שלא בכח,0.000113,0.000065,-0.000047,1.072962e-03
6,החזקת סמים שלא לצריכה עצמית,0.000559,0.000797,0.000238,1.081853e-03
7,עבירות נגד המחשב,0.000173,0.000116,-0.000056,1.747108e-03
8,עבירות המסכנות את בריאות הצבור,0.000055,0.000014,-0.000041,2.009543e-03
